# RADIO-CLIP & SigLIP 2 with TAO Toolkit

Train Adapt Optimize (TAO) Toolkit is a Python-based AI toolkit for customizing purpose-built AI models with your own data.

This notebook shows how to **train**, **run inference**, and **export** vision-language embedding models:
- **RADIO-CLIP**: C-RADIO v3-h image encoder with SigLIP 2 text encoder (ViT-H/16 backbone, 1024-dim CLIP-compatible embeddings).
- **SigLIP 2** (siglip2-so400m-256): Vision-language encoder with 1152-dim embeddings for text-to-image and image-to-image retrieval.

Both are suited for **object search** and **re-identification** in applications such as **transportation**, **warehouse operations**, and **public safety** (image + text embeddings).

<img align="center" src="https://d29g4g2dyqv443.cloudfront.net/sites/default/files/akamai/TAO/tlt-tao-toolkit-bring-your-own-model-diagram.png" width="1080">

## Learning Objectives

- Set up the TAO launcher environment and mount points.
- Prepare an image+caption dataset for CLIP-style training.
- **RADIO-CLIP**: Train, run inference, and export to ONNX.
- **SigLIP 2**: Train, run inference, and export to ONNX.

## Table of Contents

0. [Set up env variables and map drives](#head-0)
1. [Prepare dataset](#head-1)
2. [RADIO-CLIP: Train, inference, export](#head-radio-clip)
3. [SigLIP 2: Train, inference, export](#head-siglip2)

## 0. Set up env variables and map drives <a class="anchor" id="head-0"></a>

The TAO launcher uses Docker. Map your local data, specs, and results directories so they are visible inside the container. Configure `~/.tao_mounts.json` and set the encryption key if you use NGC pretrained weights.

**Note:** Set `LOCAL_PROJECT_DIR` to a path on your machine that will be mapped into the container.

In [ ]:
import os

# Local project directory mapped into the TAO container.
%env LOCAL_PROJECT_DIR=/path/to/local/tao-experiments

os.environ["HOST_DATA_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "data", "clip")
os.environ["HOST_RESULTS_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "clip_results")
os.environ["HOST_SPECS_DIR"] = os.path.join(os.getenv("NOTEBOOK_ROOT", os.getcwd()), "specs")

# Set your NGC API key if using encrypted pretrained weights from NGC.
%env KEY=nvidia_tao

In [ ]:
! mkdir -p $HOST_DATA_DIR/train/images $HOST_DATA_DIR/train/captions
! mkdir -p $HOST_DATA_DIR/inference/images
! mkdir -p $HOST_SPECS_DIR
! mkdir -p $HOST_RESULTS_DIR

In [ ]:
import json

mounts_file = os.path.expanduser("~/.tao_mounts.json")
tlt_configs = {
    "Mounts": [
        {
            "source": os.environ["LOCAL_PROJECT_DIR"],
            "destination": "/workspace/tao-experiments"
        },
        {
            "source": os.environ["HOST_DATA_DIR"],
            "destination": "/data"
        },
        {
            "source": os.environ["HOST_SPECS_DIR"],
            "destination": "/specs"
        },
        {
            "source": os.environ["HOST_RESULTS_DIR"],
            "destination": "/results"
        }
    ],
    "DockerOptions": {
        "shm_size": "16G",
        "ulimits": {
            "memlock": -1,
            "stack": 67108864
        }
    }
}
with open(mounts_file, "w") as mfile:
    json.dump(tlt_configs, mfile, indent=4)
print("Wrote", mounts_file)

In [ ]:
! cat ~/.tao_mounts.json

In [ ]:
%env DATA_DIR=/data
%env SPECS_DIR=/specs
%env RESULTS_DIR=/results

## 1. Prepare dataset <a class="anchor" id="head-1"></a>

CLIP-style training expects **image–text pairs**: for each image, a text caption file (e.g. `image_name.txt` in a captions folder).

- **Training:** Put images in `$HOST_DATA_DIR/train/images` and one caption file per image in `$HOST_DATA_DIR/train/captions` (same base name, `.txt` suffix).
- **Inference:** Put images in `$HOST_DATA_DIR/inference/images`.

The notebook maps `$HOST_DATA_DIR` to `/data` in the container. So the spec files use `/data/train/images`, `/data/train/captions`, and `/data/inference/images`. Ensure the layout under `$HOST_DATA_DIR` is:

```
  train/
    images/   <- .jpg, .png, etc.
    captions/ <- .txt per image (same base name)
  inference/
    images/   <- images for embedding extraction
```

In [ ]:
! ls -la $HOST_DATA_DIR/train/images/ | head -5
! ls -la $HOST_DATA_DIR/train/captions/ | head -5

## 2. RADIO-CLIP: Train, inference, export <a class="anchor" id="head-radio-clip"></a>

RADIO-CLIP uses the C-RADIO v3-h image encoder (ViT-H/16) with the SigLIP 2 text encoder and produces 1024-dimensional embeddings.

### 2.1 View RADIO-CLIP experiment spec

In [ ]:
! cat $HOST_SPECS_DIR/experiment_radio-clip.yaml

### 2.2 Train RADIO-CLIP

In [ ]:
! tao model clip train \
    -e $SPECS_DIR/experiment_radio-clip.yaml \
    results_dir=$RESULTS_DIR/radio-clip \
    dataset.train.datasets.0.image_dir=$DATA_DIR/train/images \
    dataset.train.datasets.0.caption_dir=$DATA_DIR/train/captions

In [ ]:
! ls -ltrh $HOST_RESULTS_DIR/radio-clip/train 2>/dev/null || echo "No train dir yet; run training above."

### 2.3 Inference with RADIO-CLIP

In [ ]:
# Set CHECKPOINT to your trained .pth (e.g. latest or best epoch).
os.environ["RADIO_CLIP_CHECKPOINT"] = os.path.join(os.getenv("HOST_RESULTS_DIR"), "radio-clip/train/clip_latest.pth")

! tao model clip inference \
    -e $SPECS_DIR/experiment_radio-clip.yaml \
    results_dir=$RESULTS_DIR/radio-clip \
    inference.checkpoint=$RESULTS_DIR/radio-clip/train/clip_latest.pth \
    inference.datasets.0.image_dir=$DATA_DIR/inference/images

### 2.4 Export RADIO-CLIP to ONNX

In [ ]:
! tao model clip export \
    -e $SPECS_DIR/experiment_radio-clip.yaml \
    export.checkpoint=$RESULTS_DIR/radio-clip/train/clip_latest.pth \
    export.onnx_file=$RESULTS_DIR/radio-clip/export/radio-clip_v1.onnx

In [ ]:
! ls -lth $HOST_RESULTS_DIR/radio-clip/export 2>/dev/null || true

## 3. SigLIP 2: Train, inference, export <a class="anchor" id="head-siglip2"></a>

SigLIP 2 (siglip2-so400m-256) is a vision-language encoder with 1152-dim embeddings. The same dataset layout is used.

### 3.1 View SigLIP 2 experiment spec

In [ ]:
! cat $HOST_SPECS_DIR/experiment_siglip2.yaml

### 3.2 Train SigLIP 2

In [ ]:
! tao model clip train \
    -e $SPECS_DIR/experiment_siglip2.yaml \
    results_dir=$RESULTS_DIR/siglip2 \
    dataset.train.datasets.0.image_dir=$DATA_DIR/train/images \
    dataset.train.datasets.0.caption_dir=$DATA_DIR/train/captions

In [ ]:
! ls -ltrh $HOST_RESULTS_DIR/siglip2/train 2>/dev/null || echo "No train dir yet; run training above."

### 3.3 Inference with SigLIP 2

In [ ]:
! tao model clip inference \
    -e $SPECS_DIR/experiment_siglip2.yaml \
    results_dir=$RESULTS_DIR/siglip2 \
    inference.checkpoint=$RESULTS_DIR/siglip2/train/clip_latest.pth \
    inference.datasets.0.image_dir=$DATA_DIR/inference/images

### 3.4 Export SigLIP 2 to ONNX

In [ ]:
! tao model clip export \
    -e $SPECS_DIR/experiment_siglip2.yaml \
    export.checkpoint=$RESULTS_DIR/siglip2/train/clip_latest.pth \
    export.onnx_file=$RESULTS_DIR/siglip2/export/siglip2_v1.onnx

In [ ]:
! ls -lth $HOST_RESULTS_DIR/siglip2/export 2>/dev/null || true

## References

- **TAO Toolkit:** [NVIDIA TAO Toolkit](https://developer.nvidia.com/tao-toolkit)
- **RADIO-CLIP / C-RADIO:** [GitHub](https://github.com/NVlabs/RADIO) | [AM-RADIO (CVPR 2024)](https://arxiv.org/abs/2312.06709)
- **SigLIP 2:** [HuggingFace](https://huggingface.co/google/siglip2-giant-opt-patch16-384) | [Blog](https://huggingface.co/blog/siglip2)
- **CLIP in TAO:** `nvidia_tao_pytorch/multimodal/clip` in [tao-pytorch](https://github.com/NVIDIA/tao_pytorch_backend)
